# Silver layer

Conformed hourly time series, one table per source. Bronze accepts data as the
source delivers it; silver makes the four sources describe the same world in the
same way, so that gold can simply join them.

## Target grain

One row per hour, keyed on a UTC timestamp, with Finnish local time carried
alongside for reporting.

The hour is chosen because weather is published hourly and cannot be made finer.
Everything else is denser and can be coarsened. **The coarsest source sets the
common grain**, otherwise the join would invent precision the data does not have.

UTC is the source of truth. `time_local` is derived from it with
`from_utc_timestamp`, never the other way round. This matters because UTC has no
daylight saving time: every UTC day has exactly 24 hours, so counting, joining
and arithmetic stay correct across the March and October clock changes. Local
time is a presentation concern and is computed last.

## Transformations applied

**Grain alignment.** Fingrid consumption and wind arrive at 15-minute
resolution. Each reading is truncated to the hour it falls in and the hour is
averaged.

**Average, not sum.** Consumption and wind production are measured in megawatts,
which is a rate of power, not a quantity of energy. The mean of four quarter-hour
readings is the average power during that hour. Summing them would overstate the
value fourfold and would only be correct if the unit were megawatt hours.

**Incomplete hours are dropped.** The extraction window does not begin or end on
an exact hour, so the first and last hours contain fewer than four readings.
Their averages would be computed from a different number of observations than
every other hour, which makes them quietly incomparable. Hours are kept only when
all four quarters are present.

**Type repair, now defensive rather than corrective.** Bronze once stored the
weather `time` column as a string, because it was built from JSON dictionaries
and JSON has no timestamp type. Bronze now declares an explicit schema, so the
column already arrives as a timestamp and the cast here is a no-op.

It is kept, and so is the null check after it. `to_timestamp` fails silently and
returns null rather than raising, so an unchecked cast can produce a full table
of empty timestamps that only surfaces much later as an empty join. A check that
costs one count and protects against a silent whole-table failure is worth
keeping even when the current input cannot trigger it.

**Unit conversion.** Open-Meteo reports wind speed in km/h. It is converted to
m/s, which is the Finnish meteorological convention and the unit used when
discussing turbine output. Bronze would be the wrong place for this because it
alters the source; gold would be too late because every consumer would have to
repeat it.

## Deliberate decisions

**Weather keeps four rows per hour.** The four observation points are not
averaged into a single national figure. Vaasa sits on the west coast where most
of Finland's wind capacity is, so a national mean would destroy exactly the
regional signal that is supposed to explain wind production. Silver preserves the
grain of the source; gold decides the business view.

**Price keeps its resolution column.** The European day-ahead market moved from
60-minute to 15-minute settlement during the covered period, so the source
contains both. The same hourly mean handles either case without special casing:
the mean of four quarters is the hourly mean, and the mean of a single hourly
value is that value. The completeness rule differs though, so a complete hour is
one 60-minute interval or four 15-minute ones. `resolution` is carried forward as
lineage, so a later reader can see why early rows are coarser.

**No rounding.** Averaging produces values such as 9255.347500000002, which is
the precision limit of binary floating point rather than an error. Silver keeps
full computational precision; rounding is a presentation decision for gold or the
report.

## Tables

| Table | Grain |
| --- | --- |
| `silver_consumption` | hourly |
| `silver_wind` | hourly |
| `silver_weather` | hourly, 4 locations |
| `silver_price` | hourly |

Row counts are not written down here. Bronze loads incrementally now, so these
tables grow with every run and any number in this cell would be wrong by
tomorrow. Each cell prints its own count.

That does not make the count worthless. When these tables were first built the
counts were predicted before the run and matched exactly, which is what turns a
row count into a test rather than a number to glance at. The prediction now
belongs in the quality checks, where it can be expressed as a rule instead of a
constant.

## Not done here

Joining the sources, and trimming to their common time window. The sources do not
end on the same hour: weather stops six days back because ERA5 publishes with
that lag, while Fingrid and the price run to within hours of now. Trimming
belongs in gold, where the business view is assembled and an inner join does it
without hand-written dates.


In [0]:
from pyspark.sql import functions as F

consumption_silver = (
    spark.table("workspace.energy_weather.bronze_consumption")
    .withColumn("time_utc", F.date_trunc("hour", F.col("startTime")))
    .groupBy("time_utc")
    .agg(
        F.avg("value").alias("consumption_mw"),
        # How many 15-minute readings actually landed in this hour
        F.count("value").alias("reading_count"),
    )
    # An hour is only complete with all four quarters present.
    # This drops the partial hours at both edges of the extraction window.
    .filter(F.col("reading_count") == 4)
    .drop("reading_count")
    .withColumn(
        "time_local", F.from_utc_timestamp(F.col("time_utc"), "Europe/Helsinki")
    )
)

print("Rows:", consumption_silver.count())
consumption_silver.orderBy("time_utc").show(3, truncate=False)

In [0]:
consumption_silver.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.energy_weather.silver_consumption"
)

print("Written:", spark.table("workspace.energy_weather.silver_consumption").count())

In [0]:
wind_silver = (
    spark.table("workspace.energy_weather.bronze_wind")
    .withColumn("time_utc", F.date_trunc("hour", F.col("startTime")))
    .groupBy("time_utc")
    .agg(
        F.avg("value").alias("wind_mw"),
        F.count("value").alias("reading_count"),
    )
    .filter(F.col("reading_count") == 4)
    .drop("reading_count")
    .withColumn(
        "time_local", F.from_utc_timestamp(F.col("time_utc"), "Europe/Helsinki")
    )
)

print("Rows:", wind_silver.count())

wind_silver.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.energy_weather.silver_wind"
)

In [0]:
weather_silver = (
    spark.table("workspace.energy_weather.bronze_weather")
    # Bronze declares this column as a timestamp, so the cast is a no-op.
    # Kept as a guard: to_timestamp returns null instead of raising, and the
    # null check below would catch a source change that breaks the format.
    .withColumn("time_utc", F.to_timestamp(F.col("time")))
    .withColumn(
        "time_local", F.from_utc_timestamp(F.col("time_utc"), "Europe/Helsinki")
    )
    # Open-Meteo reports wind in km/h; m/s is the Finnish convention
    # and the unit used when discussing turbine output.
    .withColumn("wind_speed_ms", F.col("wind_speed_10m") / 3.6)
    .select(
        "time_utc",
        "time_local",
        "area_id",
        "city",
        F.col("temperature_2m").alias("temperature_c"),
        "wind_speed_ms",
    )
)

print("Rows:", weather_silver.count())
print("Null timestamps:", weather_silver.filter(F.col("time_utc").isNull()).count())


In [0]:
weather_silver.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.energy_weather.silver_weather"
)

In [0]:
price_silver = (
    spark.table("workspace.energy_weather.bronze_price")
    # Bronze declares this column as a timestamp. The cast is a guard against
    # a schema change upstream, not a repair of the current input.
    .withColumn("time_ts", F.to_timestamp(F.col("time")))
    .withColumn("time_utc", F.date_trunc("hour", F.col("time_ts")))
    .groupBy("time_utc")
    .agg(
        # Works for both resolutions: the mean of four quarters is the hourly
        # mean, and the mean of a single hourly value is that value
        F.avg("price_eur_mwh").alias("price_eur_mwh"),
        F.count("price_eur_mwh").alias("interval_count"),
        # All intervals within one hour share a resolution, so min is
        # deterministic and equal to any of them
        F.min("resolution").alias("resolution"),
    )
    # A complete hour is one 60-minute interval or four 15-minute ones
    .filter(F.col("interval_count").isin(1, 4))
    .drop("interval_count")
    .withColumn(
        "time_local", F.from_utc_timestamp(F.col("time_utc"), "Europe/Helsinki")
    )
)

print("Rows:", price_silver.count())
print("Null timestamps:", price_silver.filter(F.col("time_utc").isNull()).count())


In [0]:
price_silver.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.energy_weather.silver_price"
)